# Idle detection

> Windows idle-time measurement based on system tick counts for the most recent user input.

This module defines the `ctypes` structure required by the Windows
`GetLastInputInfo` API and maintains a reusable buffer for reading the most
recent input tick count.

The idle duration is calculated by comparing that tick count with the current
system-uptime tick count. The foreground tracker uses this value to distinguish
unattended time from active application use; the module does not capture input
content.

In [ ]:
#| default_exp idle_detector

## Windows API binding

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ctypes
from ctypes import wintypes

In [ ]:
#| export
import sys
def _require_windows():
    if sys.platform != "win32":
        raise RuntimeError("Windows idle detection is only available on Windows")

In [ ]:
#| exports
class LASTINPUTINFO(ctypes.Structure):
    "ctypes representation of the Windows `LASTINPUTINFO` structure."

    _fields_ = [
        ("cbSize", wintypes.UINT),   # Size of this structure in bytes
        ("dwTime", wintypes.DWORD),  # Tick count when the last input occurred
    ]


## Idle-time queries

In [ ]:
#| export
def get_last_input_tick():
    "Call `GetLastInputInfo` and return its last-input tick count in milliseconds."
    _require_windows()

    info = LASTINPUTINFO()
    info.cbSize = ctypes.sizeof(info)
    
    ok = ctypes.windll.user32.GetLastInputInfo(ctypes.byref(info))
    if not ok:
        raise ctypes.WinError()
    return info.dwTime

In [ ]:
#| export
def get_idle_seconds():
    "Return the elapsed seconds between system uptime and the last input tick."
    _require_windows()
    current_tick = ctypes.windll.kernel32.GetTickCount64()
    last_input_tick = get_last_input_tick()
    return (current_tick - last_input_tick)/1000

- `GetLastInputInfo` returns a success flag, but the value assigned to `ok` is
  not checked. If the API call fails, `get_last_input_tick()` may return an
  unchanged or initial `info.dwTime` value.

- `GetTickCount64()` returns a 64-bit tick count, while
  `LASTINPUTINFO.dwTime` is a 32-bit `DWORD`. The last-input count wraps after
  approximately 49.7 days of uptime, so the direct subtraction may eventually
  produce an incorrect idle duration.

- The module-level `info` structure is mutated on every API call. Confirm
  whether concurrent calls are possible and whether the shared buffer needs
  synchronization or per-call allocation.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()